# Keyframe Selection — Demo Exploration

Loads one LIBERO demo, runs all four extractors, and compares velocity
profiles, 3D trajectories, and compression ratios.

In [ ]:
# Cell 1 — Setup
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

DEMO_PATH = "~/keyframe_selection/LIBERO/libero/datasets/libero_spatial/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate_demo.hdf5"

In [ ]:
# Cell 2 — Load and print summary
from src.utils.loader import load_libero_demo, list_demos

demo = load_libero_demo(DEMO_PATH, demo_idx=0)

print(f"task_name : {demo['task_name']}")
print(f"n_demos   : {demo['n_demos']}")
print()
for key, val in demo.items():
    if key in ("n_demos", "task_name"):
        continue
    arr = np.asarray(val)
    print(f"  {key:<16} shape={str(arr.shape):<20} dtype={arr.dtype}")

In [ ]:
# Cell 3 — Run all 4 extractors
from src.extractors import (
    UniformExtractor,
    VelocityZeroExtractor,
    GripperStateExtractor,
    AWEExtractor,
)
from src.extractors.base import KeyframeExtractor

T = len(demo["ee_pos"])

# ── VelocityZeroExtractor: three configurations compared ─────────────────
vel_old      = VelocityZeroExtractor(threshold=0.02)   # old default — wrong for this task
vel_fixed    = VelocityZeroExtractor(threshold=0.005)  # fixed good value for LIBERO spatial
vel_adaptive = VelocityZeroExtractor()                 # p25 adaptive — canonical going forward

print("VelocityZeroExtractor configurations:")
print(f"  {'Config':<24} {'N':>4} {'CR':>7}  threshold_used  indices")
print("  " + "-" * 78)
for label, ext in [("threshold=0.02 (old)",      vel_old),
                   ("threshold=0.005 (fixed)",    vel_fixed),
                   ("adaptive p25 (default)",     vel_adaptive)]:
    kf = ext.extract(demo["ee_vel"])
    cr = KeyframeExtractor.compression_ratio(T, len(kf))
    print(f"  {label:<24} {len(kf):>4} {cr:>7.3f}  {ext.threshold_used:.5f}         {kf.tolist()}")

print()
print(
    f"Adaptive threshold_used = {vel_adaptive.threshold_used:.5f}  "
    f"(verify: np.percentile(speed, 25) = "
    f"{np.percentile(np.linalg.norm(demo['ee_vel'], axis=1), 25):.5f})"
)

# ── Canonical extractor set — velocity uses adaptive from here on ────────
print()
uniform  = UniformExtractor(n_keyframes=10)
velocity = VelocityZeroExtractor()              # adaptive p25
gripper  = GripperStateExtractor(min_dist=3)
awe      = AWEExtractor(error_threshold=0.01)

extractors = [
    ("uniform",   uniform,  demo["ee_pos"]),
    ("velocity",  velocity, demo["ee_vel"]),
    ("gripper",   gripper,  demo["gripper_state"]),
    ("awe",       awe,      demo["ee_pos"]),
]

results: dict[str, np.ndarray] = {}
for name, ext, traj in extractors:
    kf = ext.extract(traj)
    results[name] = kf
    cr = KeyframeExtractor.compression_ratio(T, len(kf))
    print(f"{name:<10}  indices: {kf.tolist()}")
    print(f"{'':10}  n={len(kf)}, CR={cr:.3f}")
    print()

In [ ]:
# Cell 3b — Velocity threshold diagnostic
#
# Goal: find a threshold that selects ~5–15 keyframes for a ~100-frame demo
# (10–15% compression target). The smoke test showed that threshold=0.02
# fires on every frame (EE barely exceeds 0.02 m/frame), so we need to
# understand the actual speed distribution before picking a working value.

from pathlib import Path
RESULTS = Path.cwd().parent / "results"
RESULTS.mkdir(exist_ok=True)

speed = np.linalg.norm(demo["ee_vel"], axis=1)   # (T,)
t     = np.arange(T)

# ── 1. Summary statistics ────────────────────────────────────────────────
pcts = np.percentile(speed, [10, 25, 50, 75, 90])
print("Speed statistics (m/frame):")
print(f"  min    = {speed.min():.5f}")
print(f"  max    = {speed.max():.5f}")
print(f"  mean   = {speed.mean():.5f}")
print(f"  p10    = {pcts[0]:.5f}")
print(f"  p25    = {pcts[1]:.5f}")
print(f"  p50    = {pcts[2]:.5f}")
print(f"  p75    = {pcts[3]:.5f}")
print(f"  p90    = {pcts[4]:.5f}")

# ── 2. Define candidate thresholds ───────────────────────────────────────
thresholds = [
    (0.02,      "0.02",  "tab:red",    "--"),
    (0.01,      "0.01",  "tab:orange", "--"),
    (0.005,     "0.005", "tab:green",  "--"),
    (pcts[0],   "p10",   "tab:blue",   "-."),
    (pcts[1],   "p25",   "tab:purple", "-."),
]

# ── 3. Figure: 2-row layout ───────────────────────────────────────────────
fig, (ax_hist, ax_time) = plt.subplots(
    2, 1, figsize=(12, 7),
    gridspec_kw={"height_ratios": [1, 1.6]},
)
fig.suptitle(f"Velocity threshold diagnostic — {demo['task_name']}", fontsize=11)

# Row 1: histogram
ax_hist.hist(speed, bins=50, color="steelblue", alpha=0.7, edgecolor="white", linewidth=0.4)
for val, label, color, ls in thresholds:
    ax_hist.axvline(val, color=color, linestyle=ls, linewidth=1.4,
                    label=f"{label} = {val:.4f}")
ax_hist.set_xlabel("Speed (m/frame)", fontsize=9)
ax_hist.set_ylabel("Frame count", fontsize=9)
ax_hist.set_title("Speed histogram (50 bins)", fontsize=10)
ax_hist.legend(fontsize=8, loc="upper right")
ax_hist.grid(True, alpha=0.25)

# Row 2: speed over time with horizontal threshold lines
ax_time.plot(t, speed, color="black", linewidth=1.1, alpha=0.8, label="EE speed", zorder=2)
for val, label, color, ls in thresholds:
    ax_time.axhline(val, color=color, linestyle=ls, linewidth=1.2,
                    label=f"thresh={label}")
ax_time.set_xlabel("Frame", fontsize=9)
ax_time.set_ylabel("Speed (m/frame)", fontsize=9)
ax_time.set_title("Speed over time with candidate thresholds", fontsize=10)
ax_time.legend(fontsize=8, loc="upper right")
ax_time.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(RESULTS / "threshold_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/threshold_diagnostic.png")

# ── 4. Re-run VelocityZeroExtractor at each candidate threshold ──────────
print()
print(f"{'Threshold':<12} {'N keyframes':>12} {'CR':>8}  Indices")
print("-" * 70)
for thresh in [0.02, 0.01, 0.005, pcts[0], pcts[1]]:
    label = {0.02: "0.02", 0.01: "0.01", 0.005: "0.005",
             pcts[0]: "p10", pcts[1]: "p25"}.get(thresh, f"{thresh:.4f}")
    kf = VelocityZeroExtractor(threshold=thresh, min_dist=5).extract(demo["ee_vel"])
    cr = KeyframeExtractor.compression_ratio(T, len(kf))
    in_target = "<< target" if 5 <= len(kf) <= 15 else ""
    print(f"{label:<12} {len(kf):>12} {cr:>8.3f}  {kf.tolist()}  {in_target}")

In [ ]:
# Cell 4 — Velocity profile with per-method keyframe lines
COLORS = {
    "uniform":  "tab:blue",
    "velocity": "tab:red",
    "gripper":  "tab:green",
    "awe":      "tab:orange",
}

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, speed, color="black", linewidth=1.2, alpha=0.7, label="EE speed", zorder=2)

for name, kf in results.items():
    ax.vlines(
        kf, ymin=0, ymax=speed.max() * 1.05,
        colors=COLORS[name], linewidth=0.9, alpha=0.65,
        label=f"{name} (n={len(kf)})",
    )
    ax.scatter(kf, speed[kf], color=COLORS[name], s=25, zorder=5)

ax.set_xlabel("Frame", fontsize=10)
ax.set_ylabel("Speed (m/frame)", fontsize=10)
ax.set_title(demo["task_name"], fontsize=11)
ax.legend(fontsize=8, loc="upper right")
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(RESULTS / "velocity_profile.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/velocity_profile.png")

In [ ]:
# Cell 5 — 3D trajectory with keyframe highlights
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

ee = demo["ee_pos"]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

ax.plot(ee[:, 0], ee[:, 1], ee[:, 2],
        color="lightgrey", linewidth=1.0, alpha=0.8, zorder=1, label="trajectory")

for name, kf in results.items():
    ax.scatter(
        ee[kf, 0], ee[kf, 1], ee[kf, 2],
        color=COLORS[name], s=60, zorder=5,
        label=f"{name} (n={len(kf)})", depthshade=False,
    )

ax.set_xlabel("X (m)", fontsize=9)
ax.set_ylabel("Y (m)", fontsize=9)
ax.set_zlabel("Z (m)", fontsize=9)
ax.set_title(demo["task_name"], fontsize=10)
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(RESULTS / "trajectory_3d.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/trajectory_3d.png")

In [ ]:
# Cell 6 — Comparison table
from src.eval.metrics import compression_ratio

header = f"{'Method':<12} {'N keyframes':>12} {'Compression':>13}  Indices"
print(header)
print("-" * len(header))
for name, kf in results.items():
    cr = compression_ratio(T, len(kf))
    print(f"{name:<12} {len(kf):>12} {cr:>12.3f}  {kf.tolist()}")